# 00 — Génération des données fictives

Ce notebook génère les jeux de données simulés pour le rapport d'activité cancérologie AP-HP.

- `data/aphp_data.csv` : données AP-HP + 6 GHU (2019–2023)
- `data/regional_data.csv` : données de contexte régional (Clinique, CH, CHU, PSPH)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from generate_fake_data import generate_aphp_data, generate_regional_data
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## Données AP-HP et GHU

In [ ]:
aphp_df = generate_aphp_data()
aphp_df.to_csv(DATA_DIR / 'aphp_data.csv', index=False)
print(f"OK {len(aphp_df):,} lignes -> data/aphp_data.csv")
aphp_df.head(10)

In [ ]:
# Aperçu : AP-HP total toutes pathologies
aphp_df[(aphp_df.entite == 'AP-HP') & (aphp_df.appareil == 'TOTAL')][
    ['annee','nb_patients','nb_nouveaux_patients',
     'nb_sejours_chirurgie','nb_sejours_chimiotherapie',
     'nb_sejours_radiotherapie','nb_sejours_palliatifs']
]

In [ ]:
# Distribution des appareils pour la dernière année
last = aphp_df[(aphp_df.entite == 'AP-HP') & (aphp_df.appareil != 'TOTAL') & (aphp_df.organe == 'TOTAL') & (aphp_df.annee == 2023)]
last[['appareil','nb_patients']].sort_values('nb_patients', ascending=False).reset_index(drop=True)

## Données régionales

In [ ]:
regional_df = generate_regional_data()
regional_df.to_csv(DATA_DIR / 'regional_data.csv', index=False)
print(f"OK {len(regional_df):,} lignes -> data/regional_data.csv")
regional_df.head(10)

In [ ]:
# Aperçu : totaux par type d'établissement en 2023
regional_df[(regional_df.appareil == 'TOTAL') & (regional_df.annee == 2023)][
    ['type_etab','nb_patients','nb_nouveaux_patients',
     'nb_sejours_chirurgie','nb_sejours_chimiotherapie']
].sort_values('nb_patients', ascending=False).reset_index(drop=True)

## Validation : cohérence des données

In [ ]:
import plotly.express as px

d = aphp_df[(aphp_df.entite == 'AP-HP') & (aphp_df.appareil == 'TOTAL')]
fig = px.bar(d, x='annee', y='nb_patients',
             title='AP-HP — Total patients par année',
             labels={'annee': 'Année', 'nb_patients': 'Patients'},
             color_discrete_sequence=['#003189'])
fig.show()